# 11.8 - Deep Research Agents

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds a deep-research agent end to end: it turns one hard question into a cited, structured report by looping through decompose -> plan -> search -> score sources -> extract -> verify citations -> synthesize -> stop on budget. Every stage runs with a scripted "LLM" and a mock "web" so the whole architecture executes with no API key, and two illustrative cells show the real search APIs and the multi-agent orchestrator.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install dependencies

One-time environment prep before any of the loop runs. Almost every cell in this notebook is self-contained and keyless, so this install is only needed for the two illustrative cells that call real services (Anthropic and the search APIs).

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


**Explanation**

Installs the Anthropic SDK and python-dotenv; the line is commented out because the runnable cells need nothing beyond the Python standard library.

**How the code works - step by step**
- **`anthropic>=0.40.0`** - the Anthropic Python SDK, used only by the illustrative multi-agent cell (Concept 8).
- **`python-dotenv`** - lets you load API keys from a local `.env` file instead of hardcoding them.
- The `# noqa` and leading `#` keep it inert on a normal run; uncomment it on Colab or a fresh environment.

**In one line:** optional install - the scripted cells run on the standard library alone.

**What you'll see:** (no output - environment prep)

## Setup - load API keys

Reads any provider key from the environment rather than baking it into the notebook. Any one key is enough to start; the multi-provider demos read better with two or more, but nothing here fails without a key.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


**Explanation**

A safety-and-config cell, not a model call: it seeds an empty `ANTHROPIC_API_KEY` in the environment if one is not already set, so the illustrative cells have a variable to read without crashing on a missing key.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - sets the key only if it is not already present, so an existing real key is never overwritten.
- The comments stress the rule: never hardcode keys; load them from the environment or a `.env` file.

**In one line:** wire up the key from the environment without ever hardcoding it.

**What you'll see:** (no output - environment prep)

## 1 - The deep-research loop: what it actually IS

Start with the whole shape, because every later cell is one piece of it. A deep-research agent takes one hard question and runs a pipeline - decompose into sub-questions, search sources, extract findings, cross-check for contradictions, synthesize a cited report - and the crucial idea is that it learns from what the sources actually say, not from the model's own confidence.

In [ ]:
# THE DEEP-RESEARCH LOOP - what a research agent IS (vs a search engine).
# A search engine returns LINKS. A deep-research agent returns a CITED ANSWER by
# looping: decompose -> search -> extract -> cross-check -> synthesize. We script
# the "LLM" and the "web" so the whole SHAPE runs with no API key.

def fake_llm_decompose(question):             # THINK: split into focused sub-questions
    return ["what does it cover", "what does it cost", "what career outcomes", "what do reviews say"]

WEB = {                                        # a tiny mock corpus (source -> text)
    "cover":    [("netsetos.com", "18 modules, 150 hours, hands-on projects.")],
    "cost":     [("netsetos.com", "14999 INR."), ("coursera.org", "similar track ~24500 INR.")],
    "career":   [("linkedin.com", "GenAI roles 15-40 LPA in India.")],
    "reviews":  [("reddit.com", "practical, project-heavy."), ("reddit.com", "self-paced may be enough.")],
}

def search(sub_q):                             # ACT: return sources for a sub-question
    for key, hits in WEB.items():
        if key in sub_q or key[:4] in sub_q:
            return hits
    return []

def extract(sources):                          # pull findings + note contradictions
    findings = [f"{s}: {t}" for s, t in sources]
    contradictions = [t for s, t in sources if "self-paced" in t]   # a source that pushes back
    return findings, contradictions

def synthesize(question, findings, contradictions):
    body = " ".join(f"[{f.split(':')[0]}]" for f in findings)       # a cited report stub
    note = " Contradiction noted." if contradictions else ""
    return f"Report on '{question}': grounded in {body}.{note}"

def research(question):
    subs = fake_llm_decompose(question)
    all_sources, all_findings, all_contra = [], [], []
    for sq in subs:
        src = search(sq)
        all_sources += src
        f, c = extract(src)
        all_findings += f; all_contra += c
    report = synthesize(question, all_findings, all_contra)
    return subs, all_sources, all_findings, all_contra, report

q = "Is the GenAI course worth it for a working professional?"
subs, sources, findings, contra, report = research(q)
print("Deep-research loop (decompose -> search -> extract -> synthesize):")
print(f"  sub-questions : {len(subs)}")
print(f"  sources found : {len(sources)}")
print(f"  findings      : {len(findings)}  | contradictions: {len(contra)}")
print(f"  report        : {report}")
print("A search engine would hand you the links; the agent hands you the answer.")
# Output:
# Deep-research loop (decompose -> search -> extract -> synthesize):
#   sub-questions : 4
#   sources found : 6
#   findings      : 6  | contradictions: 1
#   report        : Report on 'Is the GenAI course worth it for a working professional?': grounded in [netsetos.com] [netsetos.com] [coursera.org] [linkedin.com] [reddit.com] [reddit.com]. Contradiction noted.
# A search engine would hand you the links; the agent hands you the answer.

**Explanation**

This is the entire loop in miniature, run against a scripted `fake_llm` and a tiny mock `WEB` dictionary so the shape executes with no API key. Read it as: a search engine returns links; this pipeline returns a cited answer, and each helper here becomes a full concept later.

**How the code works - step by step**
- **`fake_llm_decompose`** - the THINK step: splits the question into four focused sub-questions (cover, cost, career, reviews).
- **`WEB` + `search`** - the ACT step: a mock corpus mapping keys to (source, text) hits, and a lookup that returns hits for a sub-question.
- **`extract`** - pulls findings from the sources and flags any that push back (the `self-paced` line) as a contradiction.
- **`synthesize`** - stitches the findings into a cited report stub, appending a note when a contradiction was found.
- **`research`** - the driver: loops every sub-question through search -> extract, accumulates sources/findings/contradictions, then synthesizes.

**In one line:** decompose -> search -> extract -> cross-check -> synthesize, all scripted so the shape runs keyless.

**What you'll see:** Prints the loop's tallies - 4 sub-questions, 6 sources found, 6 findings with 1 contradiction - then the assembled report string citing netsetos.com, coursera.org, linkedin.com and reddit.com with "Contradiction noted.", closing with the line that the agent hands you the answer, not the links.

## 2 - Decomposition and planning: parallel or sequential?

Decomposition hides a planning decision. Some sub-questions are independent and should run in parallel for speed; others are dependent - the second query needs the first answer - and must run in order. Parallelize a dependent pair and you waste tokens searching for something you cannot search yet.

In [ ]:
# DECOMPOSE + PLAN - not every sub-question can run in parallel.
# INDEPENDENT sub-questions (compare X vs Y) run in PARALLEL; DEPENDENT ones
# (find the CEO, THEN research their background) must run in ORDER. Route by dependency.

SUBQUESTIONS = [
    {"id": 1, "q": "what does the course cover",        "depends_on": None},
    {"id": 2, "q": "what does it cost vs alternatives", "depends_on": None},
    {"id": 3, "q": "what career outcomes do grads get", "depends_on": None},
    {"id": 4, "q": "for THOSE outcomes, the salary range", "depends_on": 3},   # needs #3 first
]

def plan(subs):
    parallel = [s for s in subs if s["depends_on"] is None]
    dependent = [s for s in subs if s["depends_on"] is not None]
    return parallel, dependent

def run(subs):
    parallel, dependent = plan(subs)
    order = []
    order.append(("PARALLEL", [s["id"] for s in parallel]))   # fan out independent sub-qs at once
    for s in dependent:                                       # then the ones that had to wait
        order.append(("AFTER #%d" % s["depends_on"], [s["id"]]))
    return order, len(parallel), len(dependent)

order, n_par, n_dep = run(SUBQUESTIONS)
print("Plan the sub-questions by dependency:")
for stage, ids in order:
    print(f"  {stage:10} -> sub-questions {ids}")
print(f"\n{n_par} run in parallel, {n_dep} waits on its dependency.")
naive = len(SUBQUESTIONS)              # sequential-only would run all 4 one after another
smart = 1 + n_dep                      # 1 parallel wave + the dependent tail
print(f"Sequential-only: {naive} waves. Dependency-aware: {smart} waves (parallel where safe).")
# Output:
# Plan the sub-questions by dependency:
#   PARALLEL   -> sub-questions [1, 2, 3]
#   AFTER #3   -> sub-questions [4]
#
# 3 run in parallel, 1 waits on its dependency.
# Sequential-only: 4 waves. Dependency-aware: 2 waves (parallel where safe).

**Explanation**

A dependency planner, not a model call: it takes a list of sub-questions annotated with `depends_on` and routes them into a parallel wave plus a dependent tail. The payoff it demonstrates is that dependency-aware scheduling finishes in fewer waves than running everything sequentially.

**How the code works - step by step**
- **`SUBQUESTIONS`** - four sub-questions, three with `depends_on: None` and one (salary range) that needs sub-question #3 first.
- **`plan`** - splits the list into the independent (`depends_on is None`) set and the dependent set.
- **`run`** - emits one PARALLEL stage for all independent ids, then an `AFTER #n` stage for each dependent one that had to wait.
- The `naive` vs `smart` wave count contrasts sequential-only (4 waves) with dependency-aware scheduling (1 parallel wave + the dependent tail).

**In one line:** fan out independent sub-questions at once, chain only the ones that truly must wait.

**What you'll see:** Prints the plan - PARALLEL -> sub-questions [1, 2, 3] then AFTER #3 -> [4] - reports "3 run in parallel, 1 waits on its dependency", and shows sequential-only would take 4 waves versus 2 for the dependency-aware plan.

## 3 - Source evaluation: not all sources are equal

Once you have search results, the part that separates a good agent from a plausible-sounding one is source evaluation. A peer-reviewed study and a random five-year-old blog are not equal evidence. Score each source by a credibility tier times a recency factor, and drop whatever falls below the bar before it can reach synthesis.

In [ ]:
# SOURCE EVALUATION - not all sources are equal. Score each by a CREDIBILITY
# TIER (a .gov study outranks a random blog) and RECENCY (fresh beats stale),
# then keep only what clears the bar. This is what stops a confident wrong report.

TODAY = 2026
TIER = {"gov": 5, "journal": 5, "news": 3, "industry": 3, "blog": 2, "social": 1}

SOURCES = [
    {"url": "nist.gov/report",     "tier": "gov",      "year": 2026, "text": "official benchmark data"},
    {"url": "random-blog.com",     "tier": "blog",     "year": 2021, "text": "one person's old opinion"},
    {"url": "reuters.com/story",   "tier": "news",     "year": 2026, "text": "reported figures"},
    {"url": "x.com/anon",          "tier": "social",   "year": 2026, "text": "unverified claim"},
    {"url": "acm.org/paper",       "tier": "journal",  "year": 2024, "text": "peer-reviewed study"},
]

def recency(year):                              # linear decay, floored at 0.2; newer scores higher
    age = TODAY - year
    return max(0.2, 1.0 - 0.15 * age)           # newer -> closer to 1.0

def score(s):
    return round(TIER[s["tier"]] * recency(s["year"]), 2)

ranked = sorted(SOURCES, key=score, reverse=True)
KEEP = 3.0                                       # drop anything below the bar
print("Score sources by credibility tier x recency:")
for s in ranked:
    verdict = "KEEP " if score(s) >= KEEP else "DROP "
    print(f"  {verdict} {score(s):>4}  {s['tier']:<8} {s['year']}  {s['url']}")
kept = [s for s in ranked if score(s) >= KEEP]
print(f"\nKept {len(kept)} of {len(SOURCES)} sources; a fresh .gov beats a stale blog and a social post.")
# Output:
# Score sources by credibility tier x recency:
#   KEEP   5.0  gov      2026  nist.gov/report
#   KEEP   3.5  journal  2024  acm.org/paper
#   KEEP   3.0  news     2026  reuters.com/story
#   DROP   1.0  social   2026  x.com/anon
#   DROP   0.5  blog     2021  random-blog.com
#
# Kept 3 of 5 sources; a fresh .gov beats a stale blog and a social post.

**Explanation**

A scoring-and-filtering harness: it ranks a mixed set of sources by `tier x recency` and keeps only those clearing a threshold. This is the layer that stops a stale blog or an anonymous post from outvoting an actual study.

**How the code works - step by step**
- **`TIER`** - a credibility policy mapping source types to weights (gov/journal 5, news/industry 3, blog 2, social 1).
- **`recency`** - a linear decay on the source's age, floored at 0.2, so newer sources score closer to 1.0.
- **`score`** - multiplies the tier weight by the recency factor for a single blended number.
- **Ranking + `KEEP` bar** - sorts sources high-to-low and labels each KEEP or DROP against the 3.0 threshold.

**In one line:** credibility tier times recency, then drop everything below the bar.

**What you'll see:** Prints each source with its score and a KEEP/DROP verdict - the 2026 nist.gov report (5.0) and acm.org paper (3.5) and reuters story (3.0) kept, the anonymous x.com post (1.0) and 2021 blog (0.5) dropped - and closes with "Kept 3 of 5 sources".

## 4 - Search APIs for agents: the 2026 landscape

> **Needs API keys** (set TAVILY_API_KEY and/or SERPER_API_KEY) and `pip install requests` - illustrative, not run here.

Microsoft retired the Bing Search API in August 2025, so agents moved to purpose-built APIs. This cell sketches the three kinds you actually reach for: AI-native services that return LLM-ready content, cheap SERP scrapers, and readers that turn a URL into clean Markdown.

In [ ]:
# SEARCH APIS FOR AGENTS - the 2026 landscape (illustrative minimal example).
# Microsoft retired the Bing Search API in Aug 2025, so agents moved to purpose-built
# APIs. Three kinds: AI-native (extract content), SERP scrapers (snippets only),
# and readers (URL -> clean Markdown). Mix them by query type and budget.
import os, requests   # pip install requests

def tavily_search(query, k=5):        # AI-native: structured, LLM-ready, includes content
    r = requests.post("https://api.tavily.com/search", json={
        "api_key": os.environ["TAVILY_API_KEY"], "query": query,
        "search_depth": "advanced", "max_results": k, "include_raw_content": True})
    return r.json()["results"]        # [{title, url, content, score}, ...]

def serper_then_jina(query, k=5):     # BUDGET: cheap Google SERP + free Markdown reader
    serp = requests.post("https://google.serper.dev/search",
        headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
        json={"q": query, "num": k}).json()
    pages = []
    for hit in serp.get("organic", [])[:k]:
        md = requests.get("https://r.jina.ai/" + hit["link"]).text   # prepend r.jina.ai/ -> Markdown
        pages.append({"url": hit["link"], "markdown": md})
    return pages

# Route by need: Tavily/Exa for agent-ready search, Serper+Jina for cheap high-volume,
# Firecrawl for JS-heavy pages, Brave for an independent index. There is no single winner.
# Output: (illustrative minimal example - needs TAVILY_API_KEY / SERPER_API_KEY + `pip install requests`.)
# tavily_search returns [{title, url, content, score}, ...]; serper_then_jina returns
# [{url, markdown}, ...] - clean text ready for the extract + synthesize steps.

**Explanation**

An illustrative reference showing two real search pipelines - it is not executed and needs live keys. It exists to make the routing decision concrete: pick your search stack by query type and budget, because there is no single winner.

**How the code works - step by step**
- **`tavily_search`** - the AI-native path: POSTs to Tavily with `search_depth: advanced` and returns structured, LLM-ready results with raw content included.
- **`serper_then_jina`** - the budget path: a cheap Google SERP via Serper, then each result URL prepended with `r.jina.ai/` to fetch clean Markdown for free.
- The closing comment maps query types to tools: Tavily/Exa for agent-ready search, Serper+Jina for high volume, Firecrawl for JS-heavy pages, Brave for an independent index.

**In one line:** route between AI-native, SERP-scraper, and reader APIs by what each query needs.

**What you'll see:** No output - it is an illustrative reference (needs live API keys and `requests`). The docstring notes `tavily_search` returns `[{title, url, content, score}, ...]` and `serper_then_jina` returns `[{url, markdown}, ...]`, both ready for the extract and synthesize steps.

## 5 - Grounding and citation verification: the hardest problem

This is the step that decides whether the whole report is trustworthy. A large fraction of LLM citations do not support their claim - the model attaches a citation to something the source never said - and you cannot prompt your way out of it. The fix is a separate verification pass over every cited claim.

In [ ]:
# GROUNDING + CITATION VERIFICATION - the hardest problem in deep research.
# A large fraction of LLM citations do not actually support their claim. The fix is
# not a better prompt: store source chunks, then VERIFY each claim against its chunk
# and refuse to ship the ones the source never said.

CHUNKS = {                                       # chunk_id -> the exact source text
    "c1": "The course has 18 modules and 150 hours of content.",
    "c2": "Tuition is 14999 INR with EMI available.",
    "c3": "Graduates report roles paying 15 to 40 LPA in India.",
}

# What the synthesis LLM drafted: each claim points at a chunk it says supports it.
DRAFT = [
    {"claim": "It has 18 modules.",              "cites": "c1"},
    {"claim": "Tuition is 14999 INR.",           "cites": "c2"},
    {"claim": "It guarantees a 40 LPA job.",     "cites": "c3"},   # source says "report", not "guarantee"
    {"claim": "It includes 5000 GCP credits.",   "cites": "c2"},   # chunk never mentions credits
    {"claim": "It offers about 200 hours of content.", "cites": "c1"},  # chunk says 150, not 200
]

def verify(claim, chunk_text):
    # A real system uses an NLI / LLM entailment check; here a keyword-overlap stand-in.
    import re
    key = [w for w in re.findall(r"[a-z0-9]+", claim.lower()) if len(w) > 3 and w not in ("with","that","this")]
    hits = sum(1 for w in key if w in chunk_text.lower())
    if hits == len(key) and key:            return "SUPPORTED"
    if "guarantee" in claim.lower():         return "UNSUPPORTED"   # source hedges, claim over-states
    if hits >= 1:                            return "PARTIAL"
    return "UNSUPPORTED"

print("Verify every cited claim against its source chunk:")
counts = {"SUPPORTED": 0, "PARTIAL": 0, "UNSUPPORTED": 0}
for d in DRAFT:
    verdict = verify(d["claim"], CHUNKS[d["cites"]])
    counts[verdict] += 1
    print(f"  [{verdict:<11}] {d['claim']}  (cites {d['cites']})")
print(f"\nShip the {counts['SUPPORTED']} SUPPORTED; hedge the {counts['PARTIAL']} PARTIAL; drop the {counts['UNSUPPORTED']} UNSUPPORTED.")
print("Without this layer a confident, cited report can be mostly fabricated.")
# Output:
# Verify every cited claim against its source chunk:
#   [SUPPORTED  ] It has 18 modules.  (cites c1)
#   [SUPPORTED  ] Tuition is 14999 INR.  (cites c2)
#   [UNSUPPORTED] It guarantees a 40 LPA job.  (cites c3)
#   [UNSUPPORTED] It includes 5000 GCP credits.  (cites c2)
#   [PARTIAL    ] It offers about 200 hours of content.  (cites c1)
#
# Ship the 2 SUPPORTED; hedge the 1 PARTIAL; drop the 2 UNSUPPORTED.
# Without this layer a confident, cited report can be mostly fabricated.

**Explanation**

A verification harness that stands in for the third layer of the production defense (store chunks, cite chunk ids, verify): it checks each drafted claim against the exact source chunk it points at and buckets it as SUPPORTED, PARTIAL, or UNSUPPORTED. The keyword-overlap check is a keyless stand-in for what a real system does with an NLI/entailment model.

**How the code works - step by step**
- **`CHUNKS`** - the source of truth: chunk ids mapped to the exact text of each source.
- **`DRAFT`** - what the synthesis LLM produced: five claims, each naming the chunk it says supports it, including an over-stated "guarantee", an invented credits claim, and a wrong hours number.
- **`verify`** - extracts meaningful words from a claim and counts how many appear in the cited chunk; full overlap is SUPPORTED, a `guarantee` over-statement is forced UNSUPPORTED, partial overlap is PARTIAL, none is UNSUPPORTED.
- The loop tallies each verdict and prints the ship/hedge/drop rule.

**In one line:** check every claim against its cited chunk and refuse to ship the ones the source never said.

**What you'll see:** Prints a verdict per claim - the 18-modules and 14999-INR claims SUPPORTED, the "guarantees a 40 LPA job" and "5000 GCP credits" claims UNSUPPORTED, and "200 hours" against a 150-hours chunk PARTIAL - then "Ship the 2 SUPPORTED; hedge the 1 PARTIAL; drop the 2 UNSUPPORTED."

## 6 - Synthesis: map-reduce and the faithfulness tax

With grounded findings in hand, the agent writes the report. The standard pattern is map-reduce - summarize each source (map), then fold the summaries into one report (reduce). The catch is a faithfulness tax: the more abstractive the summary, the more likely it quietly changed a number, so the fix is to quote the critical facts verbatim.

In [ ]:
# SYNTHESIS: MAP-REDUCE + FAITHFULNESS - combine many sources into one report
# without drifting from the facts. MAP: summarize each source. REDUCE: synthesize.
# The trap: the more you abstract, the LESS faithful; so QUOTE critical facts verbatim.

SOURCES = [
    {"src": "netsetos.com", "text": "18 modules, 150 hours.", "fact": "150 hours"},
    {"src": "coursera.org", "text": "comparable track around 24500 INR.", "fact": "24500 INR"},
    {"src": "linkedin.com", "text": "GenAI roles pay 15 to 40 LPA in India.", "fact": "15-40 LPA"},
    {"src": "reddit.com",   "text": "learners call it practical and project-heavy.", "fact": None},
]

def map_summarize(s):                            # MAP: one short summary per source
    return {"src": s["src"], "summary": s["text"][:40], "fact": s["fact"]}

def reduce_report(summaries, abstractiveness):
    # abstractiveness 0..3: higher = more paraphrase = less faithful.
    verbatim = [m["fact"] for m in summaries if m["fact"]]        # critical facts kept EXACT
    report = "Synthesis: the course runs 150 hours; peers cost ~24500 INR; roles pay 15-40 LPA."
    faithful = "150 hours" in report and "15-40 LPA" in report    # the facts survived
    if abstractiveness >= 3:                                       # over-abstraction drops a number
        report = "Synthesis: a fairly long, mid-priced course with solid job prospects."
        faithful = "150 hours" in report
    return report, verbatim, faithful

summaries = [map_summarize(s) for s in SOURCES]
print("MAP - per-source summaries:")
for m in summaries:
    print(f"  [{m['src']}] {m['summary']}  fact={m['fact']}")

for a in (1, 3):
    report, verbatim, faithful = reduce_report(summaries, abstractiveness=a)
    print(f"\nREDUCE (abstractiveness={a}): faithful={faithful}")
    print(f"  {report}")
print("\nMore abstraction reads smoother but drops facts; extract-then-abstract keeps numbers verbatim.")
# Output:
# MAP - per-source summaries:
#   [netsetos.com] 18 modules, 150 hours.  fact=150 hours
#   [coursera.org] comparable track around 24500 INR.  fact=24500 INR
#   [linkedin.com] GenAI roles pay 15 to 40 LPA in India.  fact=15-40 LPA
#   [reddit.com] learners call it practical and project-h  fact=None
#
# REDUCE (abstractiveness=1): faithful=True
#   Synthesis: the course runs 150 hours; peers cost ~24500 INR; roles pay 15-40 LPA.
#
# REDUCE (abstractiveness=3): faithful=False
#   Synthesis: a fairly long, mid-priced course with solid job prospects.
#
# More abstraction reads smoother but drops facts; extract-then-abstract keeps numbers verbatim.

**Explanation**

A two-level demonstration of the abstraction tradeoff: it runs the same reduce step at low and high abstractiveness and shows the numbers surviving in the faithful version and vanishing in the over-paraphrased one. Extract-then-abstract - quote facts, paraphrase glue - is the lesson it makes concrete.

**How the code works - step by step**
- **`SOURCES`** - four sources, each carrying its text and the single critical `fact` to preserve.
- **`map_summarize`** - the MAP step: produces one short summary per source while keeping the `fact` attached.
- **`reduce_report`** - the REDUCE step: at low abstractiveness it keeps the numbers and passes the faithfulness check; at abstractiveness >= 3 it swaps in a smooth, number-free sentence that fails the check.
- The final loop runs reduce at abstractiveness 1 and 3 to contrast the two.

**In one line:** map summaries, reduce to one report, and watch high abstraction silently drop the facts.

**What you'll see:** Prints the per-source MAP summaries with their facts, then two REDUCE runs: abstractiveness=1 is faithful=True and keeps "150 hours" and "15-40 LPA", while abstractiveness=3 is faithful=False with a vague "fairly long, mid-priced course" sentence.

## 7 - Assemble the cited report: the payoff

This is the artifact the whole loop exists for. Take the claims that passed verification in Concept 5, give each source a stable number, and emit an executive summary, findings with inline `[n]` markers, and a numbered reference list - the actual thing a deep-research agent hands back, where every sentence traces to a source.

In [ ]:
# ASSEMBLE THE CITED REPORT - the payoff the whole loop exists for. Turn the VERIFIED
# claims (Step 4) into the structured artifact promised in the cold-open: an executive
# summary, findings with inline [n] citations, and a numbered reference list. Ship only
# what was SUPPORTED - the unsupported and partial claims never make it into the report.

SUPPORTED = [                                    # only the claims that passed verification (Step 4)
    {"text": "The course has 18 modules and 150 hours of content.", "src": "netsetos.com"},
    {"text": "Tuition is 14999 INR with EMI available.",            "src": "netsetos.com"},
    {"text": "Comparable peer tracks cost about 24500 INR.",        "src": "coursera.org"},
]

def assemble(question, claims):
    refs = []                                    # numbered, de-duplicated reference list
    def cite(src):
        if src not in refs:
            refs.append(src)
        return refs.index(src) + 1               # the [n] marker for this source
    findings = [f"{c['text']} [{cite(c['src'])}]" for c in claims]
    return {
        "summary": "A 150-hour, 14999 INR program - below comparable peers - so worth it if you will use it.",
        "findings": findings,
        "references": [f"[{i + 1}] {src}" for i, src in enumerate(refs)],
    }

report = assemble("Is the course worth it?", SUPPORTED)
print("EXECUTIVE SUMMARY:")
print("  " + report["summary"])
print("\nFINDINGS (each claim carries an inline citation):")
for line in report["findings"]:
    print("  - " + line)
print("\nREFERENCES:")
for ref in report["references"]:
    print("  " + ref)
print("\nEvery sentence traces to a numbered source - a cited, STRUCTURED report, not a link list.")
# Output:
# EXECUTIVE SUMMARY:
#   A 150-hour, 14999 INR program - below comparable peers - so worth it if you will use it.
#
# FINDINGS (each claim carries an inline citation):
#   - The course has 18 modules and 150 hours of content. [1]
#   - Tuition is 14999 INR with EMI available. [1]
#   - Comparable peer tracks cost about 24500 INR. [2]
#
# REFERENCES:
#   [1] netsetos.com
#   [2] coursera.org
#
# Every sentence traces to a numbered source - a cited, STRUCTURED report, not a link list.

**Explanation**

A report assembler that ships only the SUPPORTED claims and builds the citation apparatus automatically. It closes the arc opened in Concept 1: a cited, structured report instead of a pile of links.

**How the code works - step by step**
- **`SUPPORTED`** - only the claims that cleared verification in Concept 5; nothing unsupported or partial is present.
- **`cite`** - assigns each source a stable number the first time it appears and reuses it, de-duplicating the reference list.
- **`assemble`** - builds the three-part structure: an executive summary, findings each ending in an inline `[n]` marker, and a numbered references list.
- Printing walks the summary, findings, and references in order.

**In one line:** fold the verified claims into an executive summary, cited findings, and a numbered reference list.

**What you'll see:** Prints the finished report: an executive summary, three findings each carrying an inline citation (two marked [1] for netsetos.com, one [2] for coursera.org), and a two-entry reference list, closing with the note that every sentence traces to a numbered source.

## 8 - Multi-agent deep research: the Anthropic orchestrator-worker

> **Needs an Anthropic API key** (set ANTHROPIC_API_KEY) - illustrative, not run here.

Every commercial deep-research product runs the loop you just built; they differ in the bet. This cell sketches Anthropic's bet: a lead agent that decomposes and delegates to parallel sub-agents, each with its own clean context, plus a CitationAgent to ground the draft.

In [ ]:
# MULTI-AGENT DEEP RESEARCH - the Anthropic orchestrator-worker (illustrative minimal example).
# A lead agent decomposes and delegates to PARALLEL subagents, each with its own
# clean context; a separate CitationAgent grounds the draft. This beat a single agent
# by ~90% on Anthropic's research eval - at ~15x the tokens (the multi-agent tax, 11.5).
import anthropic, os

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def subagent(sub_question, tools):    # a worker: searches ONE sub-question in isolation
    return client.messages.create(
        model="claude-sonnet-4-6",    # cheaper workers; the lead uses a stronger model
        max_tokens=2048, tools=tools,
        messages=[{"role": "user", "content": f"Research this and report findings: {sub_question}"}])

def lead_research(question, sub_questions, tools):
    findings = [subagent(sq, tools) for sq in sub_questions]   # run workers in PARALLEL in production
    synthesis = client.messages.create(
        model="claude-opus-4-8", max_tokens=4096,              # the lead synthesises
        messages=[{"role": "user", "content": f"Question: {question}\nWorker findings: {findings}\n"
                                              "Write a cited report; a CitationAgent will verify each citation."}])
    return synthesis

# Reach for multi-agent ONLY when the sub-questions are genuinely independent (research is
# the textbook case). For a single coherent thread (like coding, 11.7), one agent wins.
# Output: (illustrative minimal example - needs anthropic + ANTHROPIC_API_KEY; run workers in parallel.)
# The lead delegates each sub-question to a subagent, then synthesises a cited report;
# a CitationAgent verifies the citations (Step 4). ~90% better than a single agent, ~15x tokens.

**Explanation**

An illustrative sketch of the orchestrator-worker pattern - it is not executed and needs a live key. It shows why multi-agent wins here (research sub-questions are genuinely independent) at the cost of the multi-agent tax: roughly 90% better than a single agent at about 15x the tokens.

**How the code works - step by step**
- **`client`** - the Anthropic client, built from the key loaded in setup.
- **`subagent`** - a worker that researches one sub-question in isolation on the cheaper `claude-sonnet-4-6` model with its own tools and clean context.
- **`lead_research`** - the orchestrator: dispatches each sub-question to a worker (parallel in production), then synthesizes on the stronger `claude-opus-4-8`, leaving a CitationAgent to verify each citation.
- The closing comment restates the rule: reach for multi-agent only when sub-questions are genuinely independent, as in research; a single coherent thread like coding stays single-agent.

**In one line:** a lead delegates independent sub-questions to parallel workers, then synthesizes a report a CitationAgent verifies.

**What you'll see:** No output - it is an illustrative reference (needs `anthropic` and ANTHROPIC_API_KEY, and the workers should run in parallel). The docstring notes the pattern is ~90% better than a single agent at ~15x the tokens.

## 9 - Cost and stopping: making it shippable

Two things separate a demo from a production deep-research agent. First, stopping criteria: the iterative search-and-refine loop has no natural end, so you stop it on coverage, novelty saturation, and a hard iteration/budget cap. Second, cost: an unoptimized query can run several dollars, and the standard stack cuts it by an order of magnitude.

In [ ]:
# COST + STOPPING - the iterative loop must know when to STOP, and cost must be
# controlled or a deep-research query runs forever and bills a fortune. Stop on
# coverage / no-new-information / a max-iteration cap; cut cost with caching + routing.

def enough(covered, total, new_findings, iteration, max_iters=4):
    if iteration >= max_iters:            return "STOP: max iterations reached"
    if covered >= total:                  return "STOP: all sub-questions covered"
    if new_findings == 0:                 return "STOP: no new information (saturated)"
    return "CONTINUE"

# Simulate the loop: each round covers more and eventually stops finding new things.
rounds = [(2, 4, 3), (4, 4, 2), (4, 4, 0)]     # (covered, total, new_findings) per round
print("Iterative loop with stopping criteria:")
for i, (cov, tot, new) in enumerate(rounds, 1):
    decision = enough(cov, tot, new, i)
    print(f"  round {i}: covered {cov}/{tot}, new findings {new} -> {decision}")
    if decision.startswith("STOP"):
        break

# Cost model for one query: naive vs the standard stack (cache + route + prompt-cache).
def cost(searches, synth_tokens_k, search_unit, cached, routed, prompt_cache):
    search_cost = searches * search_unit                          # cheap SERP vs premium AI-native
    synth_cost = synth_tokens_k * (0.20 if not routed else 0.04)  # route cheap models for sub-tasks
    if prompt_cache: synth_cost *= 0.5                            # discount on repeated context
    total = search_cost + synth_cost
    if cached: total *= 0.7                                       # semantic cache skips ~30% of queries
    return round(total, 2)

naive     = cost(searches=40, synth_tokens_k=30, search_unit=0.02,  cached=False, routed=False, prompt_cache=False)
optimized = cost(searches=40, synth_tokens_k=30, search_unit=0.002, cached=True,  routed=True,  prompt_cache=True)
print(f"\nCost per query: naive ${naive}  ->  cached+routed+prompt-cache ${optimized}"
      f"  (~{round(naive/optimized)}x cheaper)")
# Output:
# Iterative loop with stopping criteria:
#   round 1: covered 2/4, new findings 3 -> CONTINUE
#   round 2: covered 4/4, new findings 2 -> STOP: all sub-questions covered
#
# Cost per query: naive $6.8  ->  cached+routed+prompt-cache $0.48  (~14x cheaper)

**Explanation**

A combined stopping-logic and cost model, both run as small simulations. Together they turn the two production knobs - when to stop and how much it costs - into concrete numbers.

**How the code works - step by step**
- **`enough`** - the brake on the Reflexion loop: stops on max iterations, full coverage, or a round that found nothing new; otherwise CONTINUE.
- **`rounds` loop** - simulates the loop over three rounds and breaks the moment `enough` returns a STOP.
- **`cost`** - models one query's price from search count, synthesis tokens, and three levers: cheap-vs-premium search units, model routing (0.20 -> 0.04 per synth-token-k), prompt caching (halves synth cost), and semantic caching (skips ~30% of queries).
- **`naive` vs `optimized`** - runs the cost model both ways to show the order-of-magnitude drop.

**In one line:** cap the loop on coverage/novelty/iterations, then cache and route to collapse the cost.

**What you'll see:** Prints the loop stopping at round 2 ("all sub-questions covered") rather than running out of money, then the cost line: naive $6.8 dropping to $0.48 with caching + routing + prompt-caching, about 14x cheaper.

You built the full deep-research loop as a chain of small, keyless functions: decompose into sub-questions, plan them by dependency, score sources by credibility and recency, verify every citation against its source chunk, synthesize faithfully with map-reduce, assemble a numbered cited report, and cap the loop on coverage and budget. The through-line is that quality lives in the loop and its grounding layer, not in a clever prompt - a confident, well-formatted report with unverified citations is worse than no report. From here, evaluating research agents (RAGAS, faithfulness metrics) is picked up in Lesson 11.11 and Module 14, and Indian-language search in Lesson 17.3.